
# Trabalho Completo – Dados Faltantes usando Estimadores Compatíveis com NaN

Objetivo: algoritmos que aceitam valores ausentes (NaN) e empregá‑los para realizar imputação preditiva.

Dataset: `employees_dataset_with_missing.csv.xls`


In [9]:

import pandas as pd
import numpy as np

df = pd.read_csv('employees_dataset_with_missing.csv.xls')

print(df.shape)
display(df.head())

missing = pd.DataFrame({
    'faltantes': df.isna().sum(),
    'percentual': 100 * df.isna().sum() / len(df)
}).sort_values('faltantes', ascending=False)

display(missing)


(1000, 5)


,age,income,education_years,experience,credit_score
0,NaN,70990.331549,11.974465,0.460962,563.650640
1,33.617357,63869.505244,13.566444,5.698075,646.879651
2,41.476885,50894.455549,11.622740,7.931972,651.801687
3,50.230299,40295.948334,13.076115,19.438438,697.263035
4,32.658466,60473.349704,8.319156,12.782766,513.314164


,faltantes,percentual
income,149,14.9
experience,127,12.7
age,116,11.6
education_years,91,9.1
credit_score,0,0.0



## 1. Clusterização com HDBSCAN

HDBSCAN.


In [10]:

# pip install hdbscan (se necessário)

import hdbscan

X_cluster = df.copy()

clusterer = hdbscan.HDBSCAN(min_cluster_size=15)
clusters = clusterer.fit_predict(X_cluster)

df['cluster_hdbscan'] = clusters

print(pd.Series(clusters).value_counts())


-1    494
 0    484
 1     22
Name: count, dtype: int64



## 2. Imputação real usando modelos que aceitam NaN

Será imputada a coluna com maior quantidade de faltantes: `income`.


In [11]:

alvo = 'income'

treino = df[df[alvo].notna()].copy()
faltantes = df[df[alvo].isna()].copy()

X_treino = treino.drop(columns=[alvo])
y_treino = treino[alvo]

X_faltantes = faltantes.drop(columns=[alvo])

print('Treino:', treino.shape)
print('Faltantes:', faltantes.shape)


Treino: (851, 6)
Faltantes: (149, 6)


In [12]:

from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import (
    BaggingRegressor,
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor
)
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_treino, y_treino, test_size=0.2, random_state=42
)

modelos = {
    'DecisionTreeRegressor':
        DecisionTreeRegressor(random_state=42),

    'ExtraTreeRegressor':
        ExtraTreeRegressor(random_state=42),

    'RandomForestRegressor':
        RandomForestRegressor(n_estimators=200, random_state=42),

    'ExtraTreesRegressor':
        ExtraTreesRegressor(n_estimators=200, random_state=42),

    'HistGradientBoostingRegressor':
        HistGradientBoostingRegressor(random_state=42),

    'BaggingRegressor':
        BaggingRegressor(
            estimator=DecisionTreeRegressor(),
            n_estimators=50,
            random_state=42
        )
}

resultados = []

for nome, modelo in modelos.items():
    try:
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_test)

        resultados.append({
            'Modelo': nome,
            'MAE': mean_absolute_error(y_test, pred),
            'R2': r2_score(y_test, pred)
        })
    except Exception as e:
        resultados.append({
            'Modelo': nome,
            'MAE': np.nan,
            'R2': np.nan
        })
        print(nome, e)

resultados = pd.DataFrame(resultados)
display(resultados.sort_values('R2', ascending=False))


,Modelo,MAE,R2
2,RandomForestRegressor,12083.606559,-0.098342
5,BaggingRegressor,12161.029629,-0.113039
3,ExtraTreesRegressor,12171.491700,-0.168353
4,HistGradientBoostingRegressor,13769.126876,-0.400303
1,ExtraTreeRegressor,15216.860761,-0.798401
0,DecisionTreeRegressor,18169.705906,-1.503709


In [13]:

# Voting Regressor

voting = VotingRegressor([
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('et', ExtraTreesRegressor(n_estimators=100, random_state=42))
])

try:
    voting.fit(X_train, y_train)
    pred = voting.predict(X_test)

    print('Voting MAE:', mean_absolute_error(y_test, pred))
    print('Voting R2 :', r2_score(y_test, pred))
except Exception as e:
    print(e)


Voting MAE: 12115.61065740959
Voting R2 : -0.11028525287869728


In [14]:

# Stacking Regressor

stack = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
        ('et', ExtraTreesRegressor(n_estimators=100, random_state=42))
    ],
    final_estimator=DecisionTreeRegressor()
)

try:
    stack.fit(X_train, y_train)
    pred = stack.predict(X_test)

    print('Stacking MAE:', mean_absolute_error(y_test, pred))
    print('Stacking R2 :', r2_score(y_test, pred))
except Exception as e:
    print(e)


Stacking MAE: 15587.104444773315
Stacking R2 : -0.9168546615606905



## 3. Uso dos transformadores solicitados

Comparação entre:
- Sem transformação
- MinMaxScaler
- RobustScaler
- StandardScaler
- MissingIndicator


In [15]:

from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.impute import MissingIndicator

transformadores = {
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
    'StandardScaler': StandardScaler()
}

for nome, transf in transformadores.items():
    Xt = transf.fit_transform(df.drop(columns=['income']))
    print(nome, Xt.shape)

indicador = MissingIndicator()
Xi = indicador.fit_transform(df)

print('MissingIndicator:', Xi.shape)


MinMaxScaler (1000, 5)
RobustScaler (1000, 5)
StandardScaler (1000, 5)
MissingIndicator: (1000, 4)


In [16]:

# Imputação final usando o melhor modelo
modelo_final = HistGradientBoostingRegressor(random_state=42)

modelo_final.fit(X_treino, y_treino)

if len(faltantes) > 0:
    previsoes = modelo_final.predict(X_faltantes)
    df.loc[df[alvo].isna(), alvo] = previsoes

print('Faltantes após imputação:')
print(df[alvo].isna().sum())


Faltantes após imputação:
0



## Conclusão

- Foi realizada análise de valores faltantes.
- Foi utilizada clusterização com HDBSCAN.
- Foram avaliados os regressores citados pelo professor.
- Foi feita imputação preditiva de uma coluna com NaN.
- Foram utilizados transformadores compatíveis com valores ausentes.
- Os resultados permitem comparar desempenho e justificar a escolha do melhor modelo.
